# 08 — Baseline optimization

**Code under test:** `fleximorpv2/baseline_optimization.py`

**Purpose:** Smoke-test the optimizer on Alaska and inspect constraints.

Run cells **top to bottom**. Each markdown cell explains the **next code cell** — what it does, what to inspect in the output, and what counts as a pass.

Track overall audit progress in Obsidian (`FlexiMORP Calculation Audit.md`). These notebooks are the lab workbook, not the checklist.

In [ ]:
import sys
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

warnings.filterwarnings("error", category=RuntimeWarning)


def _repo_root() -> Path:
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "fleximorpv2").is_dir():
            return candidate
    raise RuntimeError(
        "Could not find fleximorp-project root. "
        "Open Jupyter from the repo or a notebooks/ subdirectory."
    )


_repo = _repo_root()
_audit = _repo / "notebooks" / "audit"
for path in (_repo, _audit):
    path_str = str(path)
    if path_str not in sys.path:
        sys.path.insert(0, path_str)

from _audit_helpers import (
    REPO_ROOT,
    OUTPUT_DIR,
    SITE_OUTPUT_DIR,
    reload_fleximorp,
    assert_close,
    assert_energy_balance,
    assert_cf_bounds,
    assert_positive,
)

reload_fleximorp()
np.random.seed(42)
print(f"Repo root: {REPO_ROOT}")
print(f"Audit outputs: {OUTPUT_DIR}")
print(f"Site outputs: {SITE_OUTPUT_DIR}")

## Step 1 — Run baseline optimize

**Run the next cell.**

Uses `target_type='production'`, `target_value=200_000`. The code comments say **kWh**; `TechnologyModel` reports **MWh/yr** — compare printed `annual_energy` to the target and note whether units align.

**Pass if:**
- Optimisation completes without exception
- LCOE > 0
- Total capacity ≤ config max (Alaska: 2 MW)
- CapEx within budget if config sets one

In [ ]:
from fleximorpv2.config import load_config
from fleximorpv2.baseline_optimization import BaselineOptimization

config = load_config("alaska")
config.uncertainty["monte_carlo_runs"] = 10
opt = BaselineOptimization(config)
results = opt.optimize("production", 200_000, method="scipy")

assert_positive(results.financial_metrics["lcoe"], label="LCOE")
print("Target production (as passed):", 200_000)
print("Annual energy (engine output):", results.technical_metrics["annual_energy"])
print("Financial:", results.financial_metrics)
print("Technical:", results.technical_metrics)
print("Capacities:", results.technology_capacities)